In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
import joblib
datapath = "../../../desktop/quant/hist/aaplIntra.csv"

In [15]:
df = pd.read_csv(datapath).copy()

In [16]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [ ]:
# 1-step ahead Close price target
df["y"] = df["Close"].shift(-1)
# Expanded lag features (percentage changes)
for k in [1, 2, 3, 5, 10, 20]:
    df[f"ret_lag_{k}"] = df["Close"].pct_change(k)
# Volatility features (rolling standard deviation)
for w in [5, 10, 20]:
    df[f"volatility_{w}"] = df["Close"].pct_change().rolling(window=w).std()
# Volume features
df["volume_norm"] = df["Volume"] / df["Volume"].rolling(window=20).mean()
df["volume_change"] = df["Volume"].pct_change()
# Price relative to moving averages
for w in [5, 10, 20]:
    df[f"close_ma_{w}"] = df["Close"] / df["Close"].rolling(window=w).mean()
# High-Low range (intraday volatility proxy)
df["hl_range"] = (df["High"] - df["Low"]) / df["Close"]
# Replace inf values with NaN, then drop all NaN rows
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
feature_cols = [c for c in df.columns if c.startswith(("ret_lag_", "volatility_", "volume_", "close_ma_")) or c == "hl_range"]
X = df[feature_cols].values.astype("float32")
y = df["y"].values.astype("float32")

In [ ]:
# train validation split (80/20)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
# Scale target (important for convergence)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel()

In [ ]:
# Elastic Net with cross-validation to find optimal hyperparameters
model = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
    alphas=np.logspace(-6, 1, 50),
    cv=5,
    max_iter=10000  # increased iterations for convergence
)
model.fit(X_train_scaled, y_train_scaled)
print(f"Best alpha: {model.alpha_:.6f}")
print(f"Best l1_ratio: {model.l1_ratio_:.2f}")
print(f"Number of features used: {np.sum(model.coef_ != 0)} / {len(model.coef_)}")
# Evaluate on training and validation sets
train_score = model.score(X_train_scaled, y_train_scaled)
val_score = model.score(X_val_scaled, y_val_scaled)
print(f"\nTraining R² score: {train_score:.6f}")
print(f"Validation R² score: {val_score:.6f}")
# Calculate MSE on original scale (inverse transform predictions)
from sklearn.metrics import mean_squared_error
train_pred = y_scaler.inverse_transform(model.predict(X_train_scaled).reshape(-1, 1)).ravel()
val_pred = y_scaler.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
train_mse = mean_squared_error(y_train, train_pred)
val_mse = mean_squared_error(y_val, val_pred)
print(f"Training MSE: {train_mse:.6f}")
print(f"Validation MSE: {val_mse:.6f}")

Best alpha: 0.001000
Best l1_ratio: 0.95
Number of features used: 7 / 15

Training R² score: 0.001132
Validation R² score: 0.000893
Training MSE: 601.290405
Validation MSE: 601.542358


In [20]:
# Save model, feature scaler, and target scaler for inference
joblib.dump({'model': model, 'scaler': scaler, 'y_scaler': y_scaler}, 'models/elasticNet.joblib')

['models/elasticNet.joblib']